# Uso de LLM con Fine-Tuning en Hugging Face 🧠

---

## 📍¿Qué es el Fine-Tuning y cuándo se utiliza?

El **Fine-Tuning** es el proceso de **ajustar un modelo preentrenado** usando un conjunto de datos específico.  
Esto permite adaptar el conocimiento general del modelo a un dominio concreto, mejorando su rendimiento en tareas personalizadas.

### 🎯 Se utiliza cuando:
- Tienes datos especializados (por ejemplo, textos médicos o legales).  
- Quieres mejorar la precisión de un modelo existente.  
- Necesitas que el modelo hable o razone como un experto en cierto tema.  

---

## 📍Proceso de Fine-Tuning con la librería 🤗 Transformers y Datasets

1️⃣ **Cargar un modelo base** (ej. `distilbert-base-uncased`).  
2️⃣ **Preparar el dataset** (texto y etiquetas).  
3️⃣ **Tokenizar** los datos (convertir texto a números).  
4️⃣ **Entrenar** el modelo con `Trainer` y `TrainingArguments`.  
5️⃣ **Evaluar y guardar** el modelo fine-tuneado.  

# 1️⃣ Cargar un modelo base y un tokenizador

In [1]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model_name = "distilbert-base-uncased"
model = DistilBertForSequenceClassification.from_pretrained(model_name)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# 2️⃣ Dataset de ejemplo en español

In [2]:
from datasets import Dataset

data = {
    "text": [
        # 🌞 Frases positivas
        "Me encanta este producto, funciona de maravilla.",
        "El servicio fue excelente y muy rápido.",
        "Estoy muy feliz con el resultado obtenido.",
        "La atención al cliente fue muy amable y eficiente.",
        "La película fue increíble, me hizo llorar de emoción.",
        "Definitivamente volvería a comprar aquí.",
        "El lugar es hermoso y muy acogedor.",
        "Mi experiencia fue fantástica, todo salió perfecto.",
        "La comida estuvo deliciosa, totalmente recomendada.",
        "Es una de las mejores decisiones que he tomado.",

        # 🌧️ Frases negativas
        "El producto llegó roto y de mala calidad.",
        "La atención fue pésima, no volveré nunca.",
        "Me siento decepcionado, esperaba mucho más.",
        "La película fue aburrida y sin sentido.",
        "El servicio fue lento y los empleados groseros."
    ],
    "label": [
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  # Positivas
        0, 0, 0, 0, 0                   # Negativas
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)
print(dataset[0])

Dataset({
    features: ['text', 'label'],
    num_rows: 15
})
{'text': 'Me encanta este producto, funciona de maravilla.', 'label': 1}


# 3️⃣ Tokenizar los datos

In [3]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

# 4️⃣ Entrenar el modelo con Trainer y TrainingArguments

In [4]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",          # output directory
    num_train_epochs=10,              # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=64,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler, optimizar el entrenamiento
    weight_decay=0.01,               # strength of weight decay
    logging_dir="./logs",            # directory for storing logs
    logging_steps=10,
)

trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=tokenized_datasets,    # training dataset
    eval_dataset=tokenized_datasets    # evaluation dataset (optional)
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [5]:
trainer.train()

Step,Training Loss
10,0.665024


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10, training_loss=0.6650241851806641, metrics={'train_runtime': 18.9693, 'train_samples_per_second': 7.907, 'train_steps_per_second': 0.527, 'total_flos': 19870109798400.0, 'train_loss': 0.6650241851806641, 'epoch': 10.0})

# guardamos el modelo

In [6]:
# ============================
# GUARDAR EL MODELO ENTRENADO
# ============================
save_directory = "./modelo_bert_finetuned"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"✅ Modelo guardado en: {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modelo guardado en: ./modelo_bert_finetuned


# cargamos nuestro modelo

In [7]:
# ============================
# CARGAR EL MODELO ENTRENADO
# ============================
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "./modelo_bert_finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

# probamos el modelo

In [8]:
# ============================
# FUNCIÓN DE PREDICCIÓN
# ============================
def predecir_texto(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    logits = outputs.logits
    prediccion = torch.argmax(logits, dim=1).item()

    # Si conoces las etiquetas, puedes mapearlas:
    etiquetas = ["negativo", "neutral", "positivo"]  # Ejemplo
    return etiquetas[prediccion] if prediccion < len(etiquetas) else prediccion



Texto: La pelicula fue la mejor que vi
Predicción: neutral


In [9]:
# ============================
# PROBAR CON UN TEXTO NUEVO
# ============================
texto_prueba = "odio la pelicula"
resultado = predecir_texto(texto_prueba)

print(f"Texto: {texto_prueba}")
print(f"Predicción: {resultado}")

Texto: odio la pelicula
Predicción: neutral
